Dynamically route logic based on input

In [1]:
!pip install langchain_core
!pip install langchain_openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.8/321.8 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.8/326.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.8 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `LangChain`, `OpenAI`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | ChatOpenAI()
    | StrOutputParser()
)


In [5]:
chain.invoke({"question": "how do I call OpenAI?"})

'OpenAI'

In [6]:
langchain_chain = PromptTemplate.from_template(
    """You are an expert in langchain. \
Always answer questions starting with "As Harrison Chase told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | ChatOpenAI()

openai_chain = PromptTemplate.from_template(
    """You are an expert in OpenAI. \
Always answer questions starting with "As Samuel Harris Altman told me". \
Respond to the following question:

Question: {question}
Answer:"""
) | ChatOpenAI()

general_chain = PromptTemplate.from_template(
    """Respond to the following question:

Question: {question}
Answer:"""
) | ChatOpenAI()

In [7]:
def route(info):
  if 'openai' in info['topic'].lower():
    return openai_chain
  elif "langchain" in info['topic'].lower():
    return langchain_chain
  else:
    return general_chain

In [8]:
# 사용자 정의 함수 wrapping >> langchain 파이프라인에서 사용
from langchain_core.runnables import RunnableLambda

# x = {"question": "how do I use OpenAI?"}
# x["question"] # >>"how do I use OpenAI?"

full_chain = {"topic": chain, "question": lambda x: x["question"]} | RunnableLambda(route)
# dictionary 에서 key 접근 x['question'] dict(key: value)


In [9]:
full_chain.invoke({"question": "how do I use OpenAI?"})

AIMessage(content='As Samuel Harris Altman told me, to use OpenAI, you can start by visiting their website and exploring the various tools and services they offer. You can also reach out to their team for assistance and guidance on how to best leverage their technology for your specific needs.', response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 46, 'total_tokens': 100}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-fac7eb10-0417-46aa-8899-3aaa659a14f9-0', usage_metadata={'input_tokens': 46, 'output_tokens': 54, 'total_tokens': 100})

In [10]:
full_chain.invoke({"question": "how do I use langchain?"})

AIMessage(content='As Harrison Chase told me, to use langchain, you first need to install the software on your device. Then, you can create a new project and start inputting text or code snippets into the program. Langchain uses natural language processing algorithms to analyze and generate code based on your inputs. You can also customize the settings and parameters to tailor the output to your specific needs.', response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 44, 'total_tokens': 120}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-d0411b26-4df1-40ec-bd54-f545476cf4a0-0', usage_metadata={'input_tokens': 44, 'output_tokens': 76, 'total_tokens': 120})

In [11]:
full_chain.invoke({"question": "what is the result of 2+2"})

AIMessage(content='The result of 2+2 is 4.', response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 27, 'total_tokens': 38}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-edd0994d-0f79-4610-abd2-08f1e1304348-0', usage_metadata={'input_tokens': 27, 'output_tokens': 11, 'total_tokens': 38})

In [12]:
# LCEL : LangChain Expression Language (LCEL)
# https://python.langchain.com/v0.1/docs/expression_language/

여기서 잠깐
- 람다 함수가 뭐죠?

In [42]:
add = lambda x, y : x + y
add(5,3)

8

In [43]:
numbers = [1,2,3,4,5]
result = list(map(lambda x: x**2, numbers))
result

[1, 4, 9, 16, 25]

Inspect your runnables

In [13]:
%pip install --upgrade --quiet  langchain langchain-openai faiss-cpu tiktoken langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 77.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.8 MB/s eta 0:00:00


In [14]:
# faiss : meta
# 그래프 이론 및 알고리즘 라이브러리

!pip install grandalf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00


In [15]:
# faiss 클래스 이용 >> 벡터 스토어 생성
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [16]:
vectorstore = FAISS.from_texts(
    ['harrison worked at L.A'], embedding= OpenAIEmbeddings()
)
# text data 이용, FAISS 벡터 스토어 생성

# 벡터 스토어 >> 검색기로 변환
retriever = vectorstore.as_retriever()

# 프롬프트 템플릿 정의
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# open ai 대화모델
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI()

In [17]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [18]:
chain.get_graph()

Graph(nodes={'37a102b2aea84848af70273a3c934912': Node(id='37a102b2aea84848af70273a3c934912', data=<class 'pydantic.v1.main.RunnableParallel<context,question>Input'>), 'a86ae49a891542609dc43b51ed52e186': Node(id='a86ae49a891542609dc43b51ed52e186', data=<class 'pydantic.v1.main.RunnableParallel<context,question>Output'>), '8be20a70adc34fe6971282c8e85d5b0f': Node(id='8be20a70adc34fe6971282c8e85d5b0f', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7baa09099660>)), 'f87b30d79258429c873a5152d78699ff': Node(id='f87b30d79258429c873a5152d78699ff', data=RunnablePassthrough()), '2bf8c990f6bd48e78b0b4efee836a9af': Node(id='2bf8c990f6bd48e78b0b4efee836a9af', data=ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template='Answer the question based only on the following context:\n{context}\n\nQuestion: {que

In [19]:
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

Create a runnable with the @chain decorator

In [29]:
%pip install --upgrade --quiet  langchain langchain-openai

In [30]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain
from langchain_openai import ChatOpenAI

In [31]:
prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("What is the subject of that joke: {joke}")

In [34]:
@chain
def custom_chain(text):
   prompt_val1= prompt1.invoke({"topic": text})
   output1 = ChatOpenAI().invoke(prompt_val1)
   parsed_output1 = StrOutputParser().invoke(output1)
   chain2 = prompt2 | ChatOpenAI() | StrOutputParser()
   return chain2.invoke({"joke": parsed_output1})

In [35]:
custom_chain.invoke("bears")

'The subject of the joke is a bear.'

Multi Chains

In [36]:
%pip install --upgrade --quiet  langchain langchain-openai

In [37]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [40]:
prompt1 = ChatPromptTemplate.from_template("what is the city {person} is from?")
prompt2 = ChatPromptTemplate.from_template(
    "what country is the city {city} in? respond in {language}")

model = ChatOpenAI()

# 첫번째 체인 : 인물의 출신 도시를 찾음
chain1 = prompt1 | model | StrOutputParser()

# 두번째 체인 : 출신 도시가 어느 나라에 있는지 지정된 언어로 변환

chain2 = (
    {"city": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser())

In [44]:
chain2.invoke({"person":"obama", "language":"Korean"})

'시카고, 일리노이스는 미국에 위치한 도시입니다.'

In [45]:
from langchain_core.runnables import RunnablePassthrough

# 색상 작성
prompt1 = ChatPromptTemplate.from_template(
    "generate a {attribute} color. Return the name of the color and nothing else."
)

# 색상에 해당하는 과일 작성
prompt2 = ChatPromptTemplate.from_template(
    "what is a fruit of color: {color}. Return the name of the fruit and nothing else:"
)

# 어떤 색상을 포함하고 있는 국기의 나라 작성
prompt3 = ChatPromptTemplate.from_template(
    "what is a country with a flag that has the color: {color}. Return the name of the country and nothing else:"
)

# 과일과 국기의 색에 대해 작성
prompt4 = ChatPromptTemplate.from_template(
    "What is the color of {fruit} and the flag of {country}?"
)

model_parser = model | StrOutputParser()

color_generator = (
      {"attribute": RunnablePassthrough()}
      | prompt1
      | {"color": model_parser}

)

color_to_fruit = prompt2 | model_parser
color_to_country = prompt3 | model_parser

question_generator = (
{"color": model_parser} | {"fruit": color_to_fruit, "country": color_to_country} | prompt4 )



In [46]:
question_generator.invoke("warm")

ChatPromptValue(messages=[HumanMessage(content='What is the color of Peach and the flag of India?')])

In [48]:
prompt = question_generator.invoke("warm")
print(prompt)

messages=[HumanMessage(content='What is the color of Papaya and the flag of Jamaica?')]


In [49]:
result = model.invoke(prompt)
print(result)

content='The color of papaya is typically orange, while the flag of Jamaica consists of black, green, and gold colors.' response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 20, 'total_tokens': 44}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-878c55f6-7410-45a7-949c-a44c09cfa2b3-0' usage_metadata={'input_tokens': 20, 'output_tokens': 24, 'total_tokens': 44}


In [50]:
print(result.content)

The color of papaya is typically orange, while the flag of Jamaica consists of black, green, and gold colors.


여기서 잠깐
- 파이썬에서 데코레이터(decorator)를 아시나요?

In [25]:
# decorator
# 기존 함수나 메소드의 동작 수정(변경), 확장해 줌
# 함수 수정을 하지 않고 추가기능 구현 가능

# @
# 함수명

# 간단 예제

def simple_decorator(func):
  def wrapper():
    print("공연 시작 전입니다.")
    func()
    print("공연 끝났네요.")
  return wrapper

def say_yo():
  print("에브리바디 쎄이, 쎄이 요오!")

In [26]:
say_yo()

에브리바디 쎄이, 쎄이 요오!


In [27]:
@simple_decorator
def say_yo():
  print("에브리바디 쎄이, 쎄이 요오!")

In [28]:
say_yo()

공연 시작 전입니다.
에브리바디 쎄이, 쎄이 요오!
공연 끝났네요.
